# 03 — Contextual Risk Prioritization

**Status: research notebook (EXPERIMENTAL).** Nothing here is wired into OffensiveX; `risk_model_v1` (`backend/src/offensivex/risk/engine.py`, `docs/security/risk-analysis.md`) **remains the authoritative risk engine**.

## Purpose
Study how vulnerability information combines with environmental context to prioritise findings — **without collapsing six distinct things into one opaque number**:

| # | Dimension | OffensiveX source today | In this notebook |
|---|---|---|---|
| 1 | **Technical severity** | `Finding.severity` (CVSS-derived band) | observed (CVSS v3.x of real CVEs) |
| 2 | **Exploit likelihood** | *deferred in `risk_model_v1`* (no data source yet) | observed KEV flag / EPSS (real) — used only in a clearly-marked research extension |
| 3 | **Asset criticality & exposure** | `Asset.criticality/exposure` (fallback `Target.default_*`) | **assumed** (scenario archetypes — never observed here) |
| 4 | **Evidence confidence / validation** | `Finding.confidence` (+ lifecycle `CONFIRMED`…) | **assumed** (scenario values) |
| 5 | **Attack-path / graph context** | weakest `AttackEdge.evidence_level` (`verified/observed/inferred/potential`) | **assumed** (scenario values) |
| 6 | **Final environmental priority** | `risk_score` ∈ [0,10] + `risk_band` + stored `risk_breakdown` | deterministic baseline re-implementation |

## The central honesty rule of this notebook
Public CVE datasets contain dimensions **1 and 2 only**. They contain **no asset criticality, no exposure, no analyst priority labels and no incident outcomes**, so *ground-truth environmental risk does not exist in them*. Therefore this notebook runs a **suitability gate** (§6). If no legitimate contextual labels are supplied it will **not train any ML model** — it builds the transparent deterministic baseline, a clearly labelled *simulated* scenario analysis and a sensitivity analysis, and documents what data would be needed. Every scenario number is marked **SIMULATED / ILLUSTRATIVE**: it demonstrates how the formula behaves, never how well it predicts real-world risk.

## Research questions
1. **RQ1.** Is the gate satisfied by any verified public dataset? (Expected: no.)
2. **RQ2.** How strongly do severity and exploit signals (KEV/EPSS) agree on real CVEs? Where do they diverge?
3. **RQ3.** How much do context factors reorder a severity-only ranking under `risk_model_v1` (*simulated* archetypes)?
4. **RQ4.** How sensitive are rankings to the formula's weights (one-at-a-time and Monte-Carlo)?
5. **RQ5.** *If* a legitimately labelled dataset is supplied, does ML beat the deterministic baseline (fairly, with ranking metrics)?

## 2. Environment and dependency setup
Nothing is installed silently. Locally: `pip install numpy pandas scikit-learn matplotlib seaborn joblib pyarrow`. SHAP is **not** used (permutation importance only).

In [ ]:
import os, sys, re, io, gc, json, gzip, math, time, hashlib, platform, warnings, random, importlib.util
import urllib.request, urllib.error
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib

try:
    from IPython.display import display, Markdown
except ImportError:  # plain-python fallback so the code also runs as a script
    display, Markdown = print, str

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 80)

# ---- environment detection (used ONLY for paths/setup, never to change the methodology) ----
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
ENVIRONMENT = "kaggle" if IN_KAGGLE else "colab" if IN_COLAB else "local"

# ---- dependency check (nothing is installed silently) ----
_REQUIRED = ["numpy", "pandas", "sklearn", "matplotlib", "seaborn", "joblib"]
_OPTIONAL = {"pyarrow": "needed only to read .parquet inputs", "shap": "optional; permutation importance is used if absent"}
_missing = [m for m in _REQUIRED if importlib.util.find_spec(m) is None]
if _missing:
    raise ImportError(
        f"Missing required packages: {_missing}. Install with: pip install numpy pandas scikit-learn matplotlib seaborn joblib pyarrow"
    )
OPTIONAL_AVAILABLE = {m: importlib.util.find_spec(m) is not None for m in _OPTIONAL}


def _dependency_versions():
    out = {"python": sys.version.split()[0], "platform": platform.platform()}
    for name in ["numpy", "pandas", "sklearn", "scipy", "matplotlib", "seaborn", "joblib", "pyarrow", "shap"]:
        try:
            out[name] = importlib.import_module(name).__version__
        except Exception:
            out[name] = None
    return out


DEPENDENCY_VERSIONS = _dependency_versions()


def resolve_default_dirs(cfg):
    """Fill OUTPUT_ROOT / CACHE_DIR when left as None. Override anytime via CONFIG or the OFFENSIVEX_ML_OUTPUT env var."""
    if cfg.get("OUTPUT_ROOT") is None:
        if os.environ.get("OFFENSIVEX_ML_OUTPUT"):
            cfg["OUTPUT_ROOT"] = os.environ["OFFENSIVEX_ML_OUTPUT"]
        elif IN_KAGGLE:
            cfg["OUTPUT_ROOT"] = "/kaggle/working/artifacts/ml-research"
        elif IN_COLAB:
            cfg["OUTPUT_ROOT"] = "/content/artifacts/ml-research"
        else:  # local Jupyter: <repo>/artifacts/ml-research (git-ignored), whether launched from the repo root or notebooks/
            base = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
            cfg["OUTPUT_ROOT"] = str(base / "artifacts" / "ml-research")
    if cfg.get("CACHE_DIR") is None:
        cfg["CACHE_DIR"] = str(Path(cfg["OUTPUT_ROOT"]).parent / "data_cache")
    return cfg


def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

## 3. Dataset sources, access instructions, licensing and limitations

| Source | Verified identifier (2026-10-14) | Provides | Does **not** provide | License / access |
|---|---|---|---|---|
| **Kaggle `afr1ste/nvd-cve-cvss-kev-vulnerabilities-2020-2026`** (`nvd_cve_records.parquet`) | 201,542 CVEs, 106 columns | CVSS v3.x severity, CWE, **KEV flag + date**, **EPSS (current)** | any asset, exposure, priority, outcome or confidence data | CC0; attach via Kaggle Input panel |
| NVD CVE JSON 2.0 feeds + CISA KEV + FIRST EPSS (official-feed route, `ALLOW_NETWORK=True`) | see notebook 02 §3 | same | same | public |
| *Optional, user-supplied:* a **contextual labels file** (`CONTEXT_LABELS_PATH`) | **none exists in the verified public sources** | would provide analyst priority labels / incident outcomes per finding | — | you must own the data and its license |

**Why public data cannot label contextual risk:** KEV says "this CVE was exploited somewhere", not "this finding on *this* asset deserves priority 1"; EPSS is a model estimate; CVSS is context-free by design (the very reason OffensiveX has an environmental risk engine — `docs/security/risk-analysis.md`). Using any of them as the "risk label" would be circular or mislabeled.

**Optional labels-file schema** (CSV or Parquet; one row per *finding*; validated by the gate):
`finding_id` (str) · `cve_id` (str, joins to NVD features) · `priority_label` (int, 0 = lowest … K = most urgent) · `asset_criticality` ∈ {low, medium, high, critical} · `asset_exposure` ∈ same · `evidence_confidence` ∈ [0,1] · `attack_graph_evidence` ∈ {none, potential, inferred, observed, verified} · `label_provenance` (free text naming who/what produced the label, e.g. `analyst_triage_2026Q1`, `incident_outcome`) · optional `labeled_at` (date, enables a chronological split).

## 4. Configuration variables
**Change if needed:** `DATASET_PATH`, `ALLOW_NETWORK`, `OUTPUT_ROOT`, and — only if you own legitimate labels — `CONTEXT_LABELS_PATH`.
**Scenario assumptions** (`SCENARIO_ARCHETYPES`, §9) are explicit, hypothetical and editable. **Risk weights** (`RISK_TABLES`, §7) mirror `risk_model_v1` and are verified against the repository engine when this notebook runs inside the repo.

In [ ]:
CONFIG = dict(
    DATASET_PATH=None, ALLOW_NETWORK=False, NVD_YEARS=list(range(2020, 2027)), OUTPUT_ROOT=None, CACHE_DIR=None,
    EXPERIMENT_DIR="risk_prioritization", FAST_DEV=False, FAST_DEV_ROWS=30_000,
    SEED=42,
    CONTEXT_LABELS_PATH=None,           # optional user-supplied labelled findings (see schema above); None => gate fails => no ML
    MIN_LABELED_ROWS=500, MIN_ROWS_PER_PRIORITY_CLASS=30, MAX_RULE_AGREEMENT=0.98, MAX_RULE_BALANCED_ACCURACY=0.95,
    FORBIDDEN_PROVENANCE=("synthetic", "simulated", "generated", "rule_based", "heuristic", "risk_model", "random"),
    TOP_K_FRACTION=0.05,                # "priority queue" size for ranking comparisons
    N_MONTE_CARLO=300, WEIGHT_PERTURBATION=0.25,   # +-25% uniform perturbation of weights, monotonic order preserved
    N_BOOTSTRAP=500, PLOT_SAMPLE_ROWS=20_000, N_JOBS=-1,
)
resolve_default_dirs(CONFIG)
set_seeds(CONFIG["SEED"])

In [ ]:
# ======================= CVSS helpers: vector parsing, official calculators, severity bands =======================
# Formulas: FIRST CVSS v3.1 specification section 7 (https://www.first.org/cvss/v3-1/specification-document),
# v3.0 (https://www.first.org/cvss/v3.0/specification-document), v2 (https://www.first.org/cvss/v2/guide).
# CVSS v4.0 scoring uses a macro-vector lookup table and is deliberately NOT re-implemented here.
CVSS_VERSIONS = {"v40": "4.0", "v31": "3.1", "v30": "3.0", "v2": "2.0"}


def parse_cvss_vector(vec):
    """'CVSS:3.1/AV:N/...' or (v2) 'AV:N/AC:L/...' -> (version, {metric: value}); None if unparsable."""
    if not isinstance(vec, str) or not vec:
        return None
    version, metrics = None, {}
    for part in vec.split("/"):
        if ":" not in part:
            return None
        k, v = part.split(":", 1)
        if k == "CVSS":
            version = v
        else:
            metrics[k] = v
    return (version or "2.0"), metrics


_W31 = {"AV": {"N": 0.85, "A": 0.62, "L": 0.55, "P": 0.2}, "AC": {"L": 0.77, "H": 0.44},
        "UI": {"N": 0.85, "R": 0.62}, "CIA": {"H": 0.56, "L": 0.22, "N": 0.0},
        "PR_U": {"N": 0.85, "L": 0.62, "H": 0.27}, "PR_C": {"N": 0.85, "L": 0.68, "H": 0.5}}


def _roundup_31(x):
    i = int(round(x * 100000))
    return i / 100000.0 if i % 10000 == 0 else (math.floor(i / 10000) + 1) / 10.0


def cvss3_base_score(vector):
    """Official CVSS v3.0/v3.1 base score from a vector string (None if the vector is invalid)."""
    p = parse_cvss_vector(vector)
    if p is None or p[0] not in ("3.0", "3.1"):
        return None
    ver, m = p
    try:
        changed = m["S"] == "C"
        iss = 1 - (1 - _W31["CIA"][m["C"]]) * (1 - _W31["CIA"][m["I"]]) * (1 - _W31["CIA"][m["A"]])
        impact = 7.52 * (iss - 0.029) - 3.25 * (iss - 0.02) ** 15 if changed else 6.42 * iss
        pr = (_W31["PR_C"] if changed else _W31["PR_U"])[m["PR"]]
        expl = 8.22 * _W31["AV"][m["AV"]] * _W31["AC"][m["AC"]] * pr * _W31["UI"][m["UI"]]
    except KeyError:
        return None
    if impact <= 0:
        return 0.0
    raw = min((impact + expl) * (1.08 if changed else 1.0), 10)
    return _roundup_31(raw) if ver == "3.1" else math.ceil(raw * 10 - 1e-9) / 10.0


def cvss2_base_score(vector):
    p = parse_cvss_vector(vector)
    if p is None or p[0] != "2.0":
        return None
    m = p[1]
    try:
        av, ac, au = {"L": 0.395, "A": 0.646, "N": 1.0}[m["AV"]], {"H": 0.35, "M": 0.61, "L": 0.71}[m["AC"]], {"M": 0.45, "S": 0.56, "N": 0.704}[m["Au"]]
        cia = {"N": 0.0, "P": 0.275, "C": 0.660}
        impact = 10.41 * (1 - (1 - cia[m["C"]]) * (1 - cia[m["I"]]) * (1 - cia[m["A"]]))
    except KeyError:
        return None
    expl = 20 * av * ac * au
    f = 0 if impact == 0 else 1.176
    return round(((0.6 * impact) + (0.4 * expl) - 1.5) * f + 1e-9, 1)


OFFICIAL_CALCULATORS = {"v31": cvss3_base_score, "v30": cvss3_base_score, "v2": cvss2_base_score}  # v40: none implemented


def cvss_severity(score, version):
    """Qualitative rating per the CVSS specification for the given version. Returns an upper-case label or None."""
    if score is None or (isinstance(score, float) and math.isnan(score)):
        return None
    if version == "v2":  # v2 has no NONE/CRITICAL band
        return "LOW" if score < 4.0 else "MEDIUM" if score < 7.0 else "HIGH"
    if score == 0.0:
        return "NONE"
    return "LOW" if score < 4.0 else "MEDIUM" if score < 7.0 else "HIGH" if score < 9.0 else "CRITICAL"


# Mapping to OffensiveX's finding severity enum (backend/src/offensivex/db/models.py::FindingSeverity).
CVSS_TO_OFFENSIVEX_SEVERITY = {"NONE": "info", "LOW": "low", "MEDIUM": "medium", "HIGH": "high", "CRITICAL": "critical"}
CVE_ID_RE = re.compile(r"^CVE-\d{4}-\d{4,}$")

In [ ]:
# ======================= dataset discovery, loading, fingerprinting =======================
# Supported inputs (auto-detected, in this order):
#   1. A "canonical table" named nvd_cve_records.parquet / .csv  (Kaggle: afr1ste/nvd-cve-cvss-kev-vulnerabilities-2020-2026)
#   2. Official NVD CVE JSON 2.0 yearly feeds named nvdcve-2.0-<year>.json.gz found locally
#   3. Official NVD feeds downloaded into CACHE_DIR -- ONLY when CONFIG["ALLOW_NETWORK"] is True
# Nothing is downloaded when a local source is found, and nothing is copied out of /kaggle/input.

SUPPORTED_SUFFIXES = (".parquet", ".csv", ".csv.gz", ".json", ".json.gz", ".jsonl", ".jsonl.gz")
NVD_FEED_URL = "https://nvd.nist.gov/feeds/json/cve/2.0/nvdcve-2.0-{year}.json.gz"
KEV_URL = "https://www.cisa.gov/sites/default/files/csv/known_exploited_vulnerabilities.csv"
EPSS_URL = "https://epss.cyentia.com/epss_scores-current.csv.gz"
KEV_NAMES = ("known_exploited_vulnerabilities.csv", "cisa_kev_catalog.csv")
EPSS_NAMES = ("epss_scores-current.csv.gz", "first_epss_current.csv")
_SKIP_DIRS = {"node_modules", "__pycache__", "artifacts", ".venv", "venv", "site-packages", "sample_data"}


class DataNotFoundError(RuntimeError):
    pass


def _roots_to_scan():
    if CONFIG["DATASET_PATH"]:
        return [Path(CONFIG["DATASET_PATH"]).expanduser()]
    roots = []
    if IN_KAGGLE:
        roots.append(Path("/kaggle/input"))
    roots += [Path(CONFIG["CACHE_DIR"]), Path.cwd(), Path.cwd() / "data"]
    if IN_COLAB:
        roots.append(Path("/content"))
    seen, out = set(), []
    for r in roots:
        if r.exists() and str(r.resolve()) not in seen:
            seen.add(str(r.resolve()))
            out.append(r)
    return out


def list_input_files(roots, max_depth=4, limit=500):
    found = []
    for root in roots:
        if root.is_file():
            found.append(root)
            continue
        base = len(root.resolve().parts)
        for dirpath, dirnames, filenames in os.walk(root):
            if len(Path(dirpath).resolve().parts) - base >= max_depth:
                dirnames[:] = []
            dirnames[:] = [d for d in dirnames if not d.startswith(".") and d not in _SKIP_DIRS]
            for fn in filenames:
                if fn.lower().endswith(SUPPORTED_SUFFIXES):
                    found.append(Path(dirpath) / fn)
            if len(found) >= limit:
                return found
    return found


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


def http_download(url, dest, tries=4, timeout=120, backoff=5):
    """Cached, retrying, rate-limit-aware download (honours Retry-After on 429/503). Atomic write."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"  cache hit: {dest.name} ({dest.stat().st_size/1e6:.1f} MB) -- delete it to refresh")
        return dest
    part = Path(str(dest) + ".part")
    last, wait = None, backoff
    for attempt in range(1, tries + 1):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "OffensiveX-ml-research/1.0 (research notebook)"})
            with urllib.request.urlopen(req, timeout=timeout) as r, open(part, "wb") as f:
                while block := r.read(1 << 20):
                    f.write(block)
            part.replace(dest)
            print(f"  downloaded {url} -> {dest.name} ({dest.stat().st_size/1e6:.1f} MB)")
            return dest
        except urllib.error.HTTPError as e:
            last = e
            if e.code in (400, 401, 403, 404):
                raise RuntimeError(f"HTTP {e.code} for {url} (not retryable)") from e
            ra = e.headers.get("Retry-After") if e.headers else None
            wait = int(ra) if ra and ra.isdigit() else backoff * attempt * (2 if e.code in (429, 503) else 1)
        except (urllib.error.URLError, TimeoutError, ConnectionError) as e:
            last, wait = e, backoff * attempt
        print(f"  attempt {attempt}/{tries} failed for {url}: {last}; retrying in {wait}s")
        time.sleep(wait)
    raise RuntimeError(f"Could not download {url} after {tries} attempts: {last}")


# ----------------------------- NVD CVE JSON 2.0 parsing -----------------------------
_CVSS_KEYS = {"v40": "cvssMetricV40", "v31": "cvssMetricV31", "v30": "cvssMetricV30", "v2": "cvssMetricV2"}
_CVSS_FIELDS = ["vector", "base_score", "base_severity", "source", "type", "exploitability_score", "impact_score"]
BASE_COLUMNS = ["cve_id", "source_identifier", "published_date", "last_modified_date", "vuln_status", "description_en",
                "reference_count", "weakness_count", "cpe_match_count", "primary_cwe_id"]
CANON_COLUMNS = BASE_COLUMNS + [f"cvss_{v}_{f}" for v in _CVSS_KEYS for f in _CVSS_FIELDS]


def _count_cpe_matches(configs):
    n = 0
    for conf in configs or []:
        stack = list(conf.get("nodes", []))
        while stack:
            node = stack.pop()
            n += len(node.get("cpeMatch", []))
            stack.extend(node.get("children", []))
    return n


def parse_nvd_record(rec):
    c = rec["cve"]
    row = {"cve_id": c.get("id"), "source_identifier": c.get("sourceIdentifier"),
           "published_date": c.get("published"), "last_modified_date": c.get("lastModified"),
           "vuln_status": c.get("vulnStatus")}
    row["description_en"] = next((d["value"] for d in c.get("descriptions", []) if d.get("lang") == "en"), None)
    row["reference_count"] = len(c.get("references", []))
    cwes = [d["value"] for w in c.get("weaknesses", []) for d in w.get("description", [])]
    row["weakness_count"] = len(set(cwes))
    row["primary_cwe_id"] = cwes[0] if cwes else None
    row["cpe_match_count"] = _count_cpe_matches(c.get("configurations"))
    metrics = c.get("metrics", {})
    for ver, key in _CVSS_KEYS.items():
        entries = metrics.get(key) or []
        m = next((e for e in entries if e.get("type") == "Primary"), entries[0] if entries else None)
        p = f"cvss_{ver}_"
        if m is None:
            continue
        d = m.get("cvssData", {})
        row[p + "vector"] = d.get("vectorString")
        row[p + "base_score"] = d.get("baseScore")
        row[p + "base_severity"] = d.get("baseSeverity") or m.get("baseSeverity")
        row[p + "source"] = m.get("source")
        row[p + "type"] = m.get("type")
        row[p + "exploitability_score"] = m.get("exploitabilityScore")
        row[p + "impact_score"] = m.get("impactScore")
    return row


def parse_nvd_feed(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        data = json.load(f)
    if data.get("format") != "NVD_CVE" or "vulnerabilities" not in data:
        raise ValueError(f"{path} is not an NVD CVE 2.0 feed (format={data.get('format')!r})")
    return pd.DataFrame([parse_nvd_record(r) for r in data["vulnerabilities"]])


def read_kev(path):
    kev = pd.read_csv(path)
    low = {c.lower(): c for c in kev.columns}
    id_col = low.get("cveid") or low.get("cve_id")
    add_col = low.get("dateadded") or low.get("date_added")
    if id_col is None or add_col is None:
        raise ValueError(f"KEV file {path} lacks cveID/dateAdded columns: {list(kev.columns)}")
    out = kev[[id_col, add_col]].rename(columns={id_col: "cve_id", add_col: "kev_dateadded"})
    out["kev_dateadded"] = pd.to_datetime(out["kev_dateadded"], errors="coerce")
    return out.drop_duplicates("cve_id")


def read_epss(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8") as f:
        header = f.readline()
    skip = 1 if header.startswith("#") else 0
    e = pd.read_csv(path, skiprows=skip)
    e.columns = [c.lower() for c in e.columns]
    if "cve" in e.columns:
        e = e.rename(columns={"cve": "cve_id"})
    meta = dict(re.findall(r"(\w+):([^,\s]+)", header)) if skip else {}
    e["epss_score_date"] = meta.get("score_date")
    e["epss_model_version"] = meta.get("model_version")
    return e.rename(columns={"epss": "epss_score", "percentile": "epss_percentile"})


def _find_named(files, names):
    lowered = {f.name.lower(): f for f in files}
    for n in names:
        if n in lowered:
            return lowered[n]
    return None


def _standardise(df):
    df = df.copy()
    for c in ("published_date", "last_modified_date"):
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce", utc=True).dt.tz_localize(None).astype("datetime64[ns]")
    if "kev_dateadded" in df.columns:
        df["kev_dateadded"] = pd.to_datetime(df["kev_dateadded"], errors="coerce")
    return df


def _read_table(path):
    p = str(path).lower()
    if p.endswith(".parquet"):
        return pd.read_parquet(path)
    if p.endswith((".csv", ".csv.gz")):
        return pd.read_csv(path, low_memory=False)
    raise ValueError(f"Unsupported file type for a canonical table: {path}")


def load_dataset(need_exploit_signals=False):
    """Returns (DataFrame in canonical schema, metadata dict). Raises DataNotFoundError with instructions."""
    roots = _roots_to_scan()
    print(f"Scanning for inputs under: {[str(r) for r in roots]}")
    files = list_input_files(roots)
    print(f"Found {len(files)} candidate data file(s):")
    for f in files[:25]:
        print(f"  {f}  ({f.stat().st_size/1e6:.1f} MB)")
    if len(files) > 25:
        print(f"  ... and {len(files) - 25} more")

    kind, sources = None, []
    canon = _find_named(files, ("nvd_cve_records.parquet", "nvd_cve_records.csv"))
    explicit = Path(CONFIG["DATASET_PATH"]).expanduser() if CONFIG["DATASET_PATH"] else None
    if canon is not None:
        kind, sources = "canonical_table", [canon]
    elif explicit is not None and explicit.is_file():
        kind, sources = "canonical_table", [explicit]
    else:
        feeds = sorted(f for f in files if (m := re.fullmatch(r"nvdcve-2\.0-(\d{4})\.json\.gz", f.name))
                       and int(m.group(1)) in set(CONFIG["NVD_YEARS"]))
        if feeds:
            kind, sources = "nvd_feeds", feeds
    if kind is None:
        if not CONFIG["ALLOW_NETWORK"]:
            raise DataNotFoundError(
                "No supported dataset found and CONFIG['ALLOW_NETWORK'] is False, so nothing will be downloaded.\n"
                "Do ONE of the following:\n"
                "  Kaggle : add the dataset 'afr1ste/nvd-cve-cvss-kev-vulnerabilities-2020-2026' via the notebook Input panel.\n"
                "  Any    : set CONFIG['DATASET_PATH'] to a folder/file containing nvd_cve_records.parquet|csv or nvdcve-2.0-<year>.json.gz.\n"
                "  Any    : set CONFIG['ALLOW_NETWORK']=True to download the official NVD yearly feeds (+KEV/EPSS) into CACHE_DIR."
            )
        print("No local dataset found; ALLOW_NETWORK=True -> downloading official NVD feeds into the cache directory.")
        sources = [http_download(NVD_FEED_URL.format(year=y), Path(CONFIG["CACHE_DIR"]) / f"nvdcve-2.0-{y}.json.gz")
                   for y in CONFIG["NVD_YEARS"]]
        kind = "nvd_feeds"
        files = files + list(sources)

    extra = []
    if kind == "nvd_feeds":
        parts = []
        for s in sources:
            t0 = time.time()
            parts.append(parse_nvd_feed(s))
            print(f"  parsed {Path(s).name}: {len(parts[-1]):,} CVEs in {time.time()-t0:.0f}s")
        df = pd.concat(parts, ignore_index=True)
        for col in CANON_COLUMNS:
            if col not in df.columns:
                df[col] = np.nan
        df = df.drop_duplicates("cve_id", keep="last")
        if need_exploit_signals:
            kev_f, epss_f = _find_named(files, KEV_NAMES), _find_named(files, EPSS_NAMES)
            if kev_f is None or epss_f is None:
                if not CONFIG["ALLOW_NETWORK"]:
                    raise DataNotFoundError(
                        "Exploit-signal files not found locally (need CISA KEV csv and FIRST EPSS csv.gz) and ALLOW_NETWORK is False.\n"
                        f"  KEV : {KEV_URL}\n  EPSS: {EPSS_URL}\nPlace them in CACHE_DIR or enable ALLOW_NETWORK."
                    )
                kev_f = kev_f or http_download(KEV_URL, Path(CONFIG["CACHE_DIR"]) / "known_exploited_vulnerabilities.csv")
                epss_f = epss_f or http_download(EPSS_URL, Path(CONFIG["CACHE_DIR"]) / "epss_scores-current.csv.gz")
            kev, epss = read_kev(kev_f), read_epss(epss_f)
            df = df.merge(kev, on="cve_id", how="left").merge(epss, on="cve_id", how="left")
            df["kev_is_known_exploited"] = df["kev_dateadded"].notna()
            extra = [kev_f, epss_f]
    else:
        df = _read_table(sources[0])
        if need_exploit_signals:
            for need in ("kev_is_known_exploited", "kev_dateadded", "epss_score"):
                if need not in df.columns:
                    raise DataNotFoundError(f"Canonical table lacks '{need}'; this notebook needs KEV/EPSS columns "
                                            "(use the afr1ste Kaggle dataset or the official-feed route with ALLOW_NETWORK).")
    df = _standardise(df)

    file_info = [{"name": Path(s).name, "bytes": Path(s).stat().st_size, "sha256": sha256_file(s)} for s in list(sources) + extra]
    fingerprint = hashlib.sha256(json.dumps([f["sha256"] for f in file_info] + [len(df)]).encode()).hexdigest()
    source_note = {
        "canonical_table": "Kaggle dataset afr1ste/nvd-cve-cvss-kev-vulnerabilities-2020-2026 (CC0 per Kaggle listing; derived from NVD/CISA KEV/FIRST EPSS)"
                           " or a user-supplied table with the same schema",
        "nvd_feeds": "Official NVD CVE JSON 2.0 yearly feeds (https://nvd.nist.gov/vuln/data-feeds) [+ CISA KEV + FIRST EPSS]",
    }[kind]
    meta = {"source_kind": kind, "source_description": source_note, "files": file_info, "dataset_fingerprint_sha256": fingerprint,
            "n_rows_loaded": int(len(df)), "n_columns": int(df.shape[1]), "environment": ENVIRONMENT,
            "loaded_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    return df, meta

In [ ]:
# ======================= output directories, plotting and artifact helpers =======================
OUTPUT_DIR = Path(CONFIG["OUTPUT_ROOT"]) / CONFIG["EXPERIMENT_DIR"]
PLOT_DIR = OUTPUT_DIR / "plots"
for _d in (OUTPUT_DIR, PLOT_DIR, Path(CONFIG["CACHE_DIR"])):
    _d.mkdir(parents=True, exist_ok=True)
SAVED_PLOTS = []
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 130, "axes.titlesize": 12, "axes.labelsize": 10})


def savefig(name, fig=None):
    """Save the figure to PLOT_DIR/<name>.png AND display it. Every plot in this notebook goes through here."""
    fig = fig or plt.gcf()
    fig.tight_layout()
    path = PLOT_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    SAVED_PLOTS.append(path.name)
    plt.show()
    plt.close(fig)
    return path


def write_json(path, obj):
    def _default(o):
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return None if np.isnan(o) else float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, (pd.Timestamp, datetime)):
            return o.isoformat()
        return str(o)
    Path(path).write_text(json.dumps(obj, indent=2, default=_default), encoding="utf-8")
    return path


_SECRET_RE = re.compile(r"(?i)(api[_-]?key|secret|passw(?:or)?d|token|authorization)\"?\s*[:=]\s*\"?[A-Za-z0-9_\-/+=]{12,}")


def scan_for_secrets(directory):
    """Fail loudly if any exported text artifact looks like it contains a credential."""
    hits = []
    for p in Path(directory).rglob("*"):
        if p.is_file() and p.suffix.lower() in {".json", ".md", ".csv", ".txt"}:
            for m in _SECRET_RE.finditer(p.read_text(encoding="utf-8", errors="ignore")):
                hits.append((p.name, m.group(0)[:40]))
    if hits:
        raise RuntimeError(f"Possible secrets found in exported artifacts: {hits}")
    return True


def export_common(tables=None, extra_config=None):
    """Writes the metadata every experiment must ship: dataset metadata+fingerprint, dependency versions, config, exclusion ledger, tables."""
    write_json(OUTPUT_DIR / "dataset_metadata.json", DATA_META)
    write_json(OUTPUT_DIR / "dependency_versions.json", DEPENDENCY_VERSIONS)
    cfg = {k: v for k, v in CONFIG.items() if k not in ("OUTPUT_ROOT", "CACHE_DIR")}
    cfg["DATASET_PATH"] = Path(cfg["DATASET_PATH"]).name if cfg.get("DATASET_PATH") else None  # no machine-specific paths in artifacts
    write_json(OUTPUT_DIR / "experiment_config.json", {**cfg, **(extra_config or {}), "environment": ENVIRONMENT})
    pd.DataFrame(EXCLUSION_LEDGER).to_csv(OUTPUT_DIR / "exclusion_ledger.csv", index=False)
    for name, t in (tables or {}).items():
        t.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)


EXCLUSION_LEDGER = []


def ledger(step, reason, before, after):
    """Record every row exclusion -- rows are never dropped silently."""
    EXCLUSION_LEDGER.append({"step": step, "reason": reason, "rows_before": int(before), "rows_removed": int(before - after), "rows_after": int(after)})
    print(f"[exclusion] {step}: {reason}: removed {before - after:,} of {before:,} -> {after:,} remain")


def bootstrap_ci(y_true, y_pred, metric, n_boot=500, seed=0, alpha=0.05, stratify=None):
    """Percentile bootstrap CI for metric(y_true, y_pred), resampling test rows (optionally stratified by a label vector)."""
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    n = len(y_true)
    idx_by = [np.where(np.asarray(stratify) == k)[0] for k in np.unique(stratify)] if stratify is not None else None
    vals = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(ix, size=len(ix), replace=True) for ix in idx_by]) if idx_by else rng.integers(0, n, n)
        try:
            vals.append(metric(y_true[idx], y_pred[idx]))
        except ValueError:
            continue
    lo, hi = np.percentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)

## 5. Dataset discovery and loading

In [ ]:
df_raw, DATA_META = load_dataset(need_exploit_signals=True)
if CONFIG["FAST_DEV"]:
    df_raw = df_raw.sample(CONFIG["FAST_DEV_ROWS"], random_state=CONFIG["SEED"]).reset_index(drop=True)
    print(f"!! FAST_DEV: random {len(df_raw):,}-row subsample. Do NOT report results from this run.")
DATA_META["fast_dev"] = CONFIG["FAST_DEV"]
print("\n================ SETUP SUMMARY ================")
print(f"environment        : {ENVIRONMENT}")
print(f"python / libs      : {DEPENDENCY_VERSIONS['python']} | numpy {DEPENDENCY_VERSIONS['numpy']} | pandas {DEPENDENCY_VERSIONS['pandas']} | sklearn {DEPENDENCY_VERSIONS['sklearn']}")
print(f"data source        : {DATA_META['source_kind']} -- {DATA_META['source_description']}")
for f in DATA_META["files"]:
    print(f"  file             : {f['name']}  {f['bytes']/1e6:.1f} MB  sha256={f['sha256'][:16]}...")
print(f"dataset fingerprint: {DATA_META['dataset_fingerprint_sha256']}")
print(f"rows x cols        : {df_raw.shape}; contextual labels file: {CONFIG['CONTEXT_LABELS_PATH'] or '(none supplied)'}")
print(f"output dir         : {OUTPUT_DIR}")
print("================================================")

REQUIRED = ["cve_id", "published_date", "vuln_status", "cvss_v31_vector", "cvss_v31_base_score", "cvss_v30_vector", "cvss_v30_base_score",
            "kev_is_known_exploited", "kev_dateadded", "epss_score", "epss_percentile", "primary_cwe_id"]
missing_cols = [c for c in REQUIRED if c not in df_raw.columns]
if missing_cols:
    raise ValueError(f"Schema mismatch: missing {missing_cols}. Expected the afr1ste Kaggle schema or the official-feed route.")
df = df_raw.copy()
n = len(df); df = df[df["cve_id"].astype(str).str.match(CVE_ID_RE)]; ledger("cve_id", "malformed CVE identifier", n, len(df))
n = len(df); df = df.sort_values("last_modified_date").drop_duplicates("cve_id", keep="last"); ledger("duplicates", "duplicate CVE id", n, len(df))
n = len(df); df = df[df["vuln_status"].fillna("") != "Rejected"]; ledger("status", "NVD vuln_status == Rejected", n, len(df))
df["cvss3_score"] = pd.to_numeric(df["cvss_v31_base_score"].where(df["cvss_v31_base_score"].notna(), df["cvss_v30_base_score"]), errors="coerce")
df["cvss3_vector"] = df["cvss_v31_vector"].where(df["cvss_v31_vector"].notna(), df["cvss_v30_vector"])
n = len(df); df = df[df["cvss3_score"].between(0, 10)]; ledger("task scope", "no CVSS v3.x score (v2/v4.0 are never mixed into v3 severity bands)", n, len(df))
calc = df["cvss3_vector"].map(cvss3_base_score)
n = len(df); df = df[calc == df["cvss3_score"]]; ledger("invalid score", "reported v3.x score disagrees with the official calculator for its own vector", n, len(df))
df["cvss3_band"] = [cvss_severity(s, "v31") for s in df["cvss3_score"]]
df["severity"] = df["cvss3_band"].map(CVSS_TO_OFFENSIVEX_SEVERITY)     # OffensiveX FindingSeverity vocabulary
df["kev"] = df["kev_is_known_exploited"].astype(int)
df["epss"] = pd.to_numeric(df["epss_score"], errors="coerce")
df = df.reset_index(drop=True)
display(pd.DataFrame(EXCLUSION_LEDGER))
print(f"analysis table: {len(df):,} CVEs ({int(df.kev.sum()):,} in KEV; EPSS available for {df.epss.notna().mean():.1%})")

## 6. Suitability gate — is there a legitimate contextual-risk label?
The gate is programmatic and conservative. A labels file passes only if **all** hold: required columns; ≥ `MIN_LABELED_ROWS` rows; ≥ 2 priority classes each with ≥ `MIN_ROWS_PER_PRIORITY_CLASS` rows; non-empty `label_provenance` that does not self-describe as synthetic/simulated/rule-generated; valid level vocabularies; and (stage 2, §13) labels that are **not (nearly) a deterministic function of `risk_model_v1`** — Spearman with the rule's output < `MAX_RULE_AGREEMENT` **and** the labels cannot be recovered from the rule's score by simple thresholds (balanced accuracy < `MAX_RULE_BALANCED_ACCURACY`; binned rule outputs have a deceptively low Spearman) — otherwise "ML vs baseline" would just re-learn the baseline. Provenance is *user-attested*; the gate cannot verify truthfulness, only reject the obvious cases.

In [ ]:
LEVELS = ["low", "medium", "high", "critical"]
GRAPH_LEVELS = ["none", "potential", "inferred", "observed", "verified"]
LABEL_COLS = ["finding_id", "cve_id", "priority_label", "asset_criticality", "asset_exposure", "evidence_confidence", "attack_graph_evidence", "label_provenance"]

candidate_labels = pd.DataFrame([
    ("kev_is_known_exploited / kev_dateadded", "CVE exploited in the wild (somewhere), per CISA", "vulnerability-level exploitation outcome; says nothing about any asset's priority", "NO (not environmental)"),
    ("epss_score / epss_percentile", "FIRST model estimate of 30-day exploitation probability", "a model output (and hindsight-contaminated for past CVEs); using it as a label = approximating EPSS", "NO (estimate, not outcome)"),
    ("cvss_* base score / severity", "context-free technical severity", "the quantity the environmental engine is meant to go beyond", "NO (circular)"),
    ("asset criticality / exposure / confidence / attack-graph evidence", "—", "do not exist in any public CVE dataset", "ABSENT"),
], columns=["public column(s)", "what it is", "why it cannot label environmental priority", "usable as contextual label?"])
display(candidate_labels)


def evaluate_gate(labels_path):
    """Stage 1. Returns (passed, reasons, labels_df | None). Never raises on bad labels -- it reports why they are unusable."""
    if not labels_path:
        return False, ["no contextual-labels file supplied (CONFIG['CONTEXT_LABELS_PATH'] is None); the verified public datasets contain no environmental outcomes or analyst priorities"], None
    p = Path(labels_path).expanduser()
    if not p.exists():
        return False, [f"labels file not found: {p}"], None
    lab = pd.read_parquet(p) if p.suffix.lower() == ".parquet" else pd.read_csv(p)
    miss = [c for c in LABEL_COLS if c not in lab.columns]
    if miss:
        return False, [f"labels file lacks required columns: {miss}"], None
    reasons = []
    if len(lab) < CONFIG["MIN_LABELED_ROWS"]:
        reasons.append(f"only {len(lab)} labelled rows (< {CONFIG['MIN_LABELED_ROWS']})")
    prov = lab["label_provenance"].fillna("").astype(str).str.strip()
    if (prov == "").mean() > 0.05:
        reasons.append("more than 5% of rows have an empty label_provenance")
    bad_prov = prov.str.lower().str.contains("|".join(CONFIG["FORBIDDEN_PROVENANCE"]), regex=True)
    if bad_prov.any():
        reasons.append(f"{int(bad_prov.sum())} rows declare synthetic/simulated/rule-generated provenance ({sorted(prov[bad_prov].unique())[:3]})")
    for c in ("asset_criticality", "asset_exposure"):
        if not lab[c].astype(str).str.lower().isin(LEVELS).all():
            reasons.append(f"{c} contains values outside {LEVELS}")
    if not lab["attack_graph_evidence"].astype(str).str.lower().isin(GRAPH_LEVELS).all():
        reasons.append(f"attack_graph_evidence contains values outside {GRAPH_LEVELS}")
    if not pd.to_numeric(lab["evidence_confidence"], errors="coerce").between(0, 1).all():
        reasons.append("evidence_confidence must be numeric within [0, 1]")
    cls = lab["priority_label"].value_counts()
    if (cls >= CONFIG["MIN_ROWS_PER_PRIORITY_CLASS"]).sum() < 2:
        reasons.append(f"fewer than 2 priority classes with >= {CONFIG['MIN_ROWS_PER_PRIORITY_CLASS']} rows (counts: {cls.to_dict()})")
    return (len(reasons) == 0), reasons, lab


GATE_PASSED, GATE_REASONS, LABELS_RAW = evaluate_gate(CONFIG["CONTEXT_LABELS_PATH"])
print(f"SUITABILITY GATE (stage 1: availability / provenance / size / vocabulary): {'PASSED' if GATE_PASSED else 'FAILED'}")
for r in GATE_REASONS:
    print("  -", r)

## 7. Deterministic baseline: `risk_model_v1`, re-implemented independently
The formula and tables below are transcribed from `docs/security/risk-analysis.md` ("`risk_model_v1` (finalized, Phase 13)") — *not* imported from the application (which needs a database session). Fairness controls: (1) **golden-value tests** with hand-computed expectations, (2) an exhaustive **monotonicity** check, (3) when run inside the repository, an **automatic parity check against the engine's source constants** (parsed with `ast`; skipped with a notice elsewhere). The baseline never sees any ML label.
```
risk = clamp( SEV_W[severity]*10 * confidence * ((IMPACT_W[criticality] + IMPACT_W[exposure]) / 2) * (1 + 0.5 * EVID_W[weakest graph edge evidence]), 0, 10 )   # graph term = 1.0 if not on the attack graph
band: critical >= 7.5, high >= 5.0, medium >= 2.5, else low
```

In [ ]:
import itertools

RISK_TABLES = {
    "model_version": "risk_model_v1",
    "severity_weight": {"info": 0.1, "low": 0.3, "medium": 0.5, "high": 0.75, "critical": 1.0},
    "impact_weight": {"low": 0.25, "medium": 0.5, "high": 0.75, "critical": 1.0},       # shared scale for asset criticality AND exposure
    "evidence_weight": {"verified": 1.0, "observed": 0.9, "inferred": 0.6, "potential": 0.3},
    "graph_bonus_coefficient": 0.5,
    "band_thresholds": {"critical": 7.5, "high": 5.0, "medium": 2.5},
}


def risk_band(score):
    t = RISK_TABLES["band_thresholds"]
    s = np.asarray(score, float)
    return np.select([s >= t["critical"], s >= t["high"], s >= t["medium"]], ["critical", "high", "medium"], "low")


def risk_v1(severity, confidence, criticality, exposure, graph_evidence="none", tables=None):
    """risk_model_v1 on scalars or equal-length array-likes of level names. graph_evidence 'none' = not on the attack graph (neutral factor 1.0)."""
    T = tables or RISK_TABLES
    sev = np.vectorize(T["severity_weight"].__getitem__)(np.atleast_1d(np.asarray(severity, dtype=object)))
    crit = np.vectorize(T["impact_weight"].__getitem__)(np.atleast_1d(np.asarray(criticality, dtype=object)))
    expo = np.vectorize(T["impact_weight"].__getitem__)(np.atleast_1d(np.asarray(exposure, dtype=object)))
    ev = np.vectorize(lambda g: 0.0 if g in (None, "none") else T["evidence_weight"][g])(np.atleast_1d(np.asarray(graph_evidence, dtype=object)))
    raw = sev * 10 * np.asarray(confidence, float) * ((crit + expo) / 2) * (1 + T["graph_bonus_coefficient"] * ev)
    return np.round(np.clip(raw, 0, 10), 2)


# (1) golden values, computed BY HAND (not by calling the function above)
GOLDEN = [  # (severity, confidence, criticality, exposure, graph) -> hand-computed score
    (("high", 0.8, "high", "critical", "verified"), 7.88),    # 7.5 * 0.8 * ((0.75+1.0)/2 = 0.875) * 1.5 = 7.875 -> 7.88
    (("critical", 1.0, "critical", "critical", "verified"), 10.0),   # 10 * 1 * 1 * 1.5 = 15 -> clamped to 10
    (("medium", 0.6, "low", "low", "none"), 0.75),            # 5 * 0.6 * 0.25 * 1.0 = 0.75
    (("info", 1.0, "high", "medium", "potential"), 0.72),     # 1 * 1 * 0.625 * 1.15 = 0.71875
    (("low", 0.8, "high", "low", "inferred"), 1.56),          # 3 * 0.8 * 0.5 * 1.3 = 1.56
]
for args, expected in GOLDEN:
    got = float(risk_v1(*[[a] for a in args])[0])
    assert abs(got - expected) <= 0.006, f"golden value mismatch for {args}: got {got}, hand-computed {expected}"

# (2) exhaustive monotonicity: raising any single input never lowers the score
sev_order, imp_order, ev_order = ["info", "low", "medium", "high", "critical"], LEVELS, GRAPH_LEVELS
conf_next = {0.2: 0.5, 0.5: 1.0, 1.0: 1.0}
grid = list(itertools.product(sev_order, [0.2, 0.5, 1.0], imp_order, imp_order, ev_order))
cols = list(zip(*grid))
scores = dict(zip(grid, risk_v1(list(cols[0]), list(cols[1]), list(cols[2]), list(cols[3]), list(cols[4]))))
violations = 0
for (s, c, cr, ex, ev), val in scores.items():
    for nxt in [(sev_order[min(sev_order.index(s) + 1, 4)], c, cr, ex, ev), (s, conf_next[c], cr, ex, ev),
                (s, c, imp_order[min(imp_order.index(cr) + 1, 3)], ex, ev), (s, c, cr, imp_order[min(imp_order.index(ex) + 1, 3)], ev),
                (s, c, cr, ex, ev_order[min(ev_order.index(ev) + 1, 4)])]:
        violations += scores[nxt] < val - 1e-9
print(f"golden-value tests passed ({len(GOLDEN)}); monotonicity checked on {len(grid):,} input combinations: {violations} violations")
assert violations == 0


# (3) parity with the repository engine source (only when this notebook runs inside the repo)
def parity_with_repository():
    import ast
    eng = None
    for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        cand = base / "backend" / "src" / "offensivex" / "risk" / "engine.py"
        if cand.exists():
            eng = cand
            break
    if eng is None:
        return None, "skipped: backend/src/offensivex/risk/engine.py not found (expected on Kaggle/Colab -- the formula is then verified by golden values only)"
    src = eng.read_text(encoding="utf-8")
    found = {}
    for node in ast.parse(src).body:
        tgt = node.target if isinstance(node, ast.AnnAssign) else (node.targets[0] if isinstance(node, ast.Assign) else None)
        if isinstance(tgt, ast.Name) and tgt.id in ("_SEVERITY_WEIGHT", "_IMPACT_WEIGHT", "_EVIDENCE_WEIGHT") and isinstance(node.value, ast.Dict):
            found[tgt.id] = {k.attr.lower(): v.value for k, v in zip(node.value.keys, node.value.values)}
    checks = {"severity_weight": ("_SEVERITY_WEIGHT", RISK_TABLES["severity_weight"]), "impact_weight": ("_IMPACT_WEIGHT", RISK_TABLES["impact_weight"]),
              "evidence_weight": ("_EVIDENCE_WEIGHT", RISK_TABLES["evidence_weight"])}
    problems = [f"{k}: engine {found.get(n)} != notebook {t}" for k, (n, t) in checks.items() if found.get(n) != t]
    if "graph_factor = 1.0 + 0.5 * evidence_multiplier" not in src:
        problems.append("graph-bonus expression changed in engine.py")
    if not all(s in src for s in ("score >= 7.5", "score >= 5.0", "score >= 2.5")):
        problems.append("band thresholds changed in engine.py")
    if 'RISK_MODEL_VERSION = "risk_model_v1"' not in src:
        problems.append("RISK_MODEL_VERSION is no longer risk_model_v1 -- update this notebook")
    return (not problems), ("PARITY OK with " + str(eng) if not problems else "PARITY FAILED: " + "; ".join(problems))


PARITY_OK, PARITY_MSG = parity_with_repository()
print(PARITY_MSG)
assert PARITY_OK in (None, True), "risk_model_v1 in the repository changed -- this notebook's baseline must be updated before use"

## 8. What is observed vs. what is assumed
Every number in the next sections is tagged **OBSERVED** (computed from real CVE records) or **SIMULATED** (computed from the explicit scenario assumptions below). *Simulated results illustrate how the formula behaves; they are not evidence about real-world risk or predictive performance.*

In [ ]:
display(pd.DataFrame([
    ("Technical severity (CVSS v3.x band)", "OBSERVED", "NVD record", "→ `Finding.severity` via CVSS_TO_OFFENSIVEX_SEVERITY (NONE→info)"),
    ("Exploit signals: KEV flag, EPSS", "OBSERVED", "CISA KEV / FIRST EPSS", "not part of `risk_model_v1` (deferred); used in a research extension only"),
    ("Asset criticality / exposure", "ASSUMED (SIMULATED)", "scenario archetypes (§11)", "→ `Asset.criticality/exposure`"),
    ("Evidence confidence", "ASSUMED (SIMULATED)", "scenario archetypes", "→ `Finding.confidence`"),
    ("Attack-graph evidence", "ASSUMED (SIMULATED)", "scenario archetypes", "→ weakest `AttackEdge.evidence_level`"),
    ("Final priority", "COMPUTED (deterministic)", "risk_model_v1 re-implementation", "→ `risk_score`, `risk_band`"),
], columns=["dimension", "status", "source", "OffensiveX mapping"]))

## 9. Severity versus exploit signals on real CVEs (OBSERVED)
*Descriptive only.* KEV and EPSS are used here to **describe** how severity and exploitation evidence relate; no model is trained and nothing is validated. EPSS for historical CVEs is hindsight-contaminated, so these plots describe the data, they do not forecast.

In [ ]:
import copy
from scipy import stats

rho = stats.spearmanr(df["cvss3_score"], df["epss"], nan_policy="omit")
band_order = [b for b in ["NONE", "LOW", "MEDIUM", "HIGH", "CRITICAL"] if b in set(df["cvss3_band"])]
by_band = df.groupby("cvss3_band").agg(n=("cve_id", "size"), kev_n=("kev", "sum"), kev_pct=("kev", lambda s: 100 * s.mean()), epss_median=("epss", "median"), epss_p90=("epss", lambda s: s.quantile(0.9)),
                                       epss_ge_0_1_pct=("epss", lambda s: 100 * (s >= 0.1).mean())).reindex(band_order)
display(by_band.round(3))
kev_df = df[df.kev == 1]
crit_df = df[df.cvss3_band == "CRITICAL"]
share_kev_leq_medium = float(kev_df["cvss3_band"].isin(["NONE", "LOW", "MEDIUM"]).mean())
share_crit_low_epss = float((crit_df["epss"] < 0.01).mean())
# severity-only review queue: how much of KEV does the top 5% by CVSS score contain? (ties broken at random with a fixed seed)
rng = np.random.default_rng(CONFIG["SEED"])
order = np.lexsort((rng.random(len(df)), -df["cvss3_score"].values))
k = int(round(CONFIG["TOP_K_FRACTION"] * len(df)))
kev_in_cvss_topk = float(df["kev"].values[order[:k]].sum() / max(df["kev"].sum(), 1))
print(f"Spearman(CVSS score, EPSS) = {rho.correlation:.3f}")
print(f"{share_kev_leq_medium:.1%} of KEV CVEs are only LOW/MEDIUM severity;  {share_crit_low_epss:.1%} of CRITICAL CVEs have EPSS < 0.01;  "
      f"a severity-only queue of the top {CONFIG['TOP_K_FRACTION']:.0%} (CVSS) contains {kev_in_cvss_topk:.1%} of KEV CVEs (chance = {CONFIG['TOP_K_FRACTION']:.0%}).")
OBSERVED = {"spearman_cvss_epss": float(rho.correlation), "kev_share_low_or_medium": share_kev_leq_medium, "critical_share_epss_lt_0_01": share_crit_low_epss, "kev_in_cvss_topk": kev_in_cvss_topk}

samp = df.sample(min(len(df), CONFIG["PLOT_SAMPLE_ROWS"] * 5), random_state=CONFIG["SEED"])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
hb = axes[0].hexbin(samp["cvss3_score"], np.log10(samp["epss"].clip(lower=1e-5)), gridsize=40, cmap="Greys", bins="log", mincnt=1); fig.colorbar(hb, ax=axes[0], label="CVE count (log)")
kk = df[df.kev == 1]
axes[0].scatter(kk["cvss3_score"] + np.random.default_rng(1).normal(0, 0.05, len(kk)), np.log10(kk["epss"].clip(lower=1e-5)), s=6, alpha=0.5, color="crimson", label=f"in KEV (n={len(kk):,})")
axes[0].set(title=f"OBSERVED: severity vs exploit likelihood (Spearman {rho.correlation:.2f})", xlabel="CVSS v3.x base score", ylabel="log10 EPSS (current snapshot)"); axes[0].legend()
bp = df.assign(logepss=np.log10(df["epss"].clip(lower=1e-5)))
sns.boxplot(data=bp, x="cvss3_band", y="logepss", order=band_order, ax=axes[1], color="#9bb7d4", fliersize=1)
ax2 = axes[1].twinx(); ax2.plot(range(len(band_order)), by_band["kev_pct"].values, "o-", color="crimson"); ax2.set_ylabel("% of band in KEV", color="crimson"); ax2.grid(False)
axes[1].set(title="OBSERVED: EPSS distribution and KEV share by severity band", xlabel="CVSS v3.x band", ylabel="log10 EPSS")
savefig("R1_severity_vs_exploit_signals_observed")

## 10. Research extension: an exploit-aware factor (hypothesis — not validated, not in OffensiveX)
`risk_model_v1` defers exploitability because no data source existed. If KEV/EPSS were adopted, the simplest *monotone* extension is a multiplier on the existing product:
`risk_ext = clamp(risk_v1_raw × (1 + W_KEV·KEV + W_EPSS·EPSS), 0, 10)` with `W_KEV = W_EPSS = 0.5` as an arbitrary placeholder. In **operation** current KEV/EPSS *are* legitimately available at scoring time (the hindsight issue only affects *evaluating* a predictive model). The weights are **judgement calls** to be set by stakeholders; §12 shows how sensitive rankings are to them. Because the extension *uses* KEV as an input, KEV can never be used to "validate" it.

In [ ]:
EXPLOIT_EXT = {"W_KEV": 0.5, "W_EPSS": 0.5, "status": "research hypothesis; NOT validated; NOT part of risk_model_v1"}
SEV_ORDER, IMP_ORDER, EV_ORDER = ["info", "low", "medium", "high", "critical"], LEVELS, GRAPH_LEVELS


def code(values, order):
    m = {v: i for i, v in enumerate(order)}
    return np.array([m[v] for v in values])


def weights_from(T):
    return (np.array([T["severity_weight"][s] for s in SEV_ORDER]), np.array([T["impact_weight"][s] for s in IMP_ORDER]),
            np.array([0.0] + [T["evidence_weight"][s] for s in EV_ORDER[1:]]), T["graph_bonus_coefficient"])


def score_fast(sev_c, conf, crit_c, exp_c, gr_c, T=None, sev_continuous=None, kev=None, epss=None, ext=None):
    """Integer-coded vectorised risk_model_v1 (+ optional research exploit factor). Same maths as risk_v1; parity asserted below."""
    sw, iw, ew, gb = weights_from(T or RISK_TABLES)
    base = (sev_continuous if sev_continuous is not None else sw[sev_c]) * 10
    raw = base * conf * ((iw[crit_c] + iw[exp_c]) / 2) * (1 + gb * ew[gr_c])
    if ext is not None:
        raw = raw * (1 + ext["W_KEV"] * kev + ext["W_EPSS"] * np.nan_to_num(epss, nan=0.0))
    return np.round(np.clip(raw, 0, 10), 2)


SEV_C = code(df["severity"], SEV_ORDER)
KEV, EPSS = df["kev"].values.astype(float), df["epss"].values
_chk = np.random.default_rng(0).integers(0, len(df), 500)
_conf = np.random.default_rng(1).uniform(0.1, 1, 500); _cr = np.random.default_rng(2).integers(0, 4, 500); _ex = np.random.default_rng(3).integers(0, 4, 500); _gr = np.random.default_rng(4).integers(0, 5, 500)
assert np.allclose(score_fast(SEV_C[_chk], _conf, _cr, _ex, _gr), risk_v1(df["severity"].values[_chk], _conf, [IMP_ORDER[i] for i in _cr], [IMP_ORDER[i] for i in _ex], [EV_ORDER[i] for i in _gr]))
print("fast scorer == readable risk_v1 on 500 random inputs")

## 11. Scenario archetypes (SIMULATED) and how context reorders a severity-only ranking
Each archetype is a **hypothetical** deployment context. For every real CVE we ask "what would `risk_model_v1` score a finding with this CVE's severity on an asset like this?". These are *not* samples of real environments and carry no frequency information.

In [ ]:
SCENARIO_ARCHETYPES = {   # EDIT FREELY -- all values are assumptions
    "A internet-facing prod database": dict(criticality="critical", exposure="critical", confidence=0.9, graph="verified"),
    "B internet-facing marketing site": dict(criticality="medium", exposure="critical", confidence=0.8, graph="observed"),
    "C isolated crown-jewel store": dict(criticality="critical", exposure="low", confidence=0.8, graph="potential"),
    "D internal dev box": dict(criticality="low", exposure="low", confidence=0.6, graph="none"),
    "E internal workstation": dict(criticality="medium", exposure="medium", confidence=0.5, graph="none"),
}
ARCH = list(SCENARIO_ARCHETYPES)
SHORT = [a.split(" ", 1)[0] for a in ARCH]
n = len(df)
SCEN = {a: score_fast(SEV_C, np.full(n, s["confidence"]), np.full(n, code([s["criticality"]], IMP_ORDER)[0]), np.full(n, code([s["exposure"]], IMP_ORDER)[0]),
                      np.full(n, code([s["graph"]], EV_ORDER)[0])) for a, s in SCENARIO_ARCHETYPES.items()}
display(pd.DataFrame(SCENARIO_ARCHETYPES).T.assign(SIMULATED="yes -- hypothetical context"))


def queue_mask(scores, frac):
    """Tie-honest priority queue: everything scoring >= the (1-frac) quantile (may exceed frac*n when scores tie)."""
    return scores >= np.quantile(scores, 1 - frac)


def jaccard(a, b):
    return float((a & b).sum() / max((a | b).sum(), 1))


cvss_cont = df["cvss3_score"].values
q_cvss = queue_mask(cvss_cont, CONFIG["TOP_K_FRACTION"])
SS = np.random.default_rng(CONFIG["SEED"]).choice(n, min(n, 30_000), replace=False)   # fixed sample for rank correlations
agree = []
for a in ARCH:
    qa = queue_mask(SCEN[a], CONFIG["TOP_K_FRACTION"])
    agree.append({"archetype": a, "SIMULATED": "yes", "kendall_tau_b_vs_CVSS_only": stats.kendalltau(cvss_cont[SS], SCEN[a][SS], variant="b").correlation,
                  "queue_size_%": 100 * qa.mean(), "queue_overlap_with_CVSS_queue (Jaccard)": jaccard(qa, q_cvss),
                  "share_of_queue_CRITICAL_%": 100 * (df["cvss3_band"].values[qa] == "CRITICAL").mean(),
                  "share_of_queue_MEDIUM_or_lower_%": 100 * np.isin(df["cvss3_band"].values[qa], ["NONE", "LOW", "MEDIUM"]).mean(),
                  "distinct_score_values": len(np.unique(SCEN[a]))})
agree_df = pd.DataFrame(agree)
display(agree_df.round(3))

# pairwise reversal: P(score(MEDIUM CVE on archetype X) > score(CRITICAL CVE on archetype Y)) -- exact, since scores depend only on the band here
med_i, crit_i = np.where(df["cvss3_band"].values == "MEDIUM")[0], np.where(df["cvss3_band"].values == "CRITICAL")[0]
rev = pd.DataFrame(index=ARCH, columns=ARCH, dtype=float)
for x in ARCH:
    for y in ARCH:
        rev.loc[x, y] = float(SCEN[x][med_i[0]] > SCEN[y][crit_i[0]]) if len(med_i) and len(crit_i) else np.nan
print("SIMULATED: does a MEDIUM-severity CVE on the row archetype outrank a CRITICAL-severity CVE on the column archetype? (1 = yes)")
display(rev.round(0))

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for a, sh in zip(ARCH, SHORT):
    sns.kdeplot(SCEN[a][SS], ax=axes[0, 0], label=sh, bw_adjust=0.6, clip=(0, 10))
axes[0, 0].set(title="SIMULATED: risk_model_v1 score distribution by archetype (all CVEs)", xlabel="risk score (0-10)", ylabel="density"); axes[0, 0].legend(title="archetype")
sev_plot_order = [s for s in SEV_ORDER if s in set(df.severity)]
long = pd.concat([pd.DataFrame({"risk": SCEN[a][SS], "severity": df["severity"].values[SS], "archetype": sh}) for a, sh in zip(ARCH, SHORT)])
sns.boxplot(data=long, x="severity", y="risk", hue="archetype", order=sev_plot_order, ax=axes[0, 1], fliersize=0)
axes[0, 1].set(title="SIMULATED: risk score by CVSS-derived severity class and archetype", xlabel="severity (FindingSeverity)", ylabel="risk score")
for ax, sv in zip(axes[1, :], ["high", "critical"]):
    grid_ = pd.DataFrame([[float(risk_v1([sv], [0.8], [c], [e], ["none"])[0]) for e in IMP_ORDER] for c in IMP_ORDER], index=IMP_ORDER, columns=IMP_ORDER)
    sns.heatmap(grid_, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0, vmax=10, ax=ax, cbar_kws={"label": "risk score"})
    ax.set(title=f"SIMULATED formula surface: severity={sv}, confidence=0.8, not on graph", xlabel="asset exposure", ylabel="asset criticality")
savefig("R2_simulated_score_distribution_and_context_surface")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
axes[0].bar(SHORT, agree_df["queue_overlap_with_CVSS_queue (Jaccard)"], color="#3b6ea5"); axes[0].set(title=f"SIMULATED: overlap of the top-{CONFIG['TOP_K_FRACTION']:.0%} queue with the CVSS-only queue", ylabel="Jaccard", ylim=(0, 1))
axes[1].bar(SHORT, agree_df["kendall_tau_b_vs_CVSS_only"], color="#5b9a5b"); axes[1].set(title="SIMULATED: rank agreement with CVSS-only ordering", ylabel="Kendall tau-b", ylim=(0, 1))
sns.heatmap(rev.astype(float), annot=True, fmt=".0f", cmap="Purples", ax=axes[2], cbar_kws={"label": "1 = medium@row outranks critical@col"}, vmin=0, vmax=1)
axes[2].set(title="SIMULATED: context reversals of severity order", xlabel="CRITICAL CVE on archetype", ylabel="MEDIUM CVE on archetype")
axes[2].set_xticklabels(SHORT); axes[2].set_yticklabels(SHORT, rotation=0)
savefig("R3_simulated_ranking_disagreement")

## 12. Sensitivity analysis of the weights (SIMULATED portfolio)
**Portfolio:** every CVE is paired with an archetype cyclically (`i mod 5`) — a deterministic, illustrative mix, *not* a real asset distribution. We measure how much the ranking changes (Kendall τ-b, top-5 % queue Jaccard, share of findings changing risk band) when:
(a) each weight is perturbed ±δ **one at a time** (order of levels preserved), (b) **all** weights are perturbed jointly (Monte-Carlo), (c) severity is made **continuous** (`CVSS/10` instead of five band weights — `risk_model_v1` creates massive ties), (d) the research exploit-weights are swept (ranking sensitivity only — *no KEV "capture" metric is shown, because KEV is an input of that variant*).

In [ ]:
arch_idx = np.arange(n) % len(ARCH)
P_conf = np.array([SCENARIO_ARCHETYPES[a]["confidence"] for a in ARCH])[arch_idx]
P_crit = np.array([code([SCENARIO_ARCHETYPES[a]["criticality"]], IMP_ORDER)[0] for a in ARCH])[arch_idx]
P_exp = np.array([code([SCENARIO_ARCHETYPES[a]["exposure"]], IMP_ORDER)[0] for a in ARCH])[arch_idx]
P_gr = np.array([code([SCENARIO_ARCHETYPES[a]["graph"]], EV_ORDER)[0] for a in ARCH])[arch_idx]
SC = np.random.default_rng(CONFIG["SEED"] + 1).choice(n, min(n, 20_000), replace=False)


def portfolio_scores(T=None, **kw):
    return score_fast(SEV_C, P_conf, P_crit, P_exp, P_gr, T=T, **kw)


BASE_P = portfolio_scores()
q_base = queue_mask(BASE_P, CONFIG["TOP_K_FRACTION"])
band_base = risk_band(BASE_P)


def compare(s):
    return {"kendall_tau_b": stats.kendalltau(BASE_P[SC], s[SC], variant="b").correlation, "queue_jaccard": jaccard(queue_mask(s, CONFIG["TOP_K_FRACTION"]), q_base),
            "band_change_%": 100 * float((risk_band(s) != band_base).mean())}


def perturb(T, key, name, factor):
    """Scale one weight, then re-impose the original ORDER of the levels so the perturbed table stays monotone."""
    T2 = copy.deepcopy(T)
    if key == "graph_bonus_coefficient":
        T2[key] = T[key] * factor
        return T2
    scaled = dict(T[key]); scaled[name] = scaled[name] * factor
    levels_in_order = sorted(T[key], key=T[key].get)
    for lvl, val in zip(levels_in_order, sorted(scaled.values())):
        T2[key][lvl] = val
    return T2


DELTA = CONFIG["WEIGHT_PERTURBATION"]
params = [(k, nm) for k in ("severity_weight", "impact_weight", "evidence_weight") for nm in RISK_TABLES[k]] + [("graph_bonus_coefficient", "graph_bonus_coefficient")]
rows = []
for key, nm in params:
    for sign in (-1, +1):
        label = "graph_bonus" if key == "graph_bonus_coefficient" else f"{key.replace('_weight', '')}:{nm}"
        rows.append({"parameter": label, "direction": f"{sign * DELTA:+.0%}", **compare(portfolio_scores(T=perturb(RISK_TABLES, key, nm, 1 + sign * DELTA)))})
oat = pd.DataFrame(rows)
display(oat.round(4))

rng_mc = np.random.default_rng(CONFIG["SEED"] + 2)
mc = []
for i in range(CONFIG["N_MONTE_CARLO"]):
    T = RISK_TABLES
    for key, nm in params:
        T = perturb(T, key, nm, rng_mc.uniform(1 - DELTA, 1 + DELTA))
    mc.append(compare(portfolio_scores(T=T)))
mc_df = pd.DataFrame(mc)
display(mc_df.describe(percentiles=[0.05, 0.5, 0.95]).round(4))

cont = portfolio_scores(sev_continuous=np.clip(df["cvss3_score"].values / 10.0, 0.01, 1.0))
gran = {"variant": "continuous severity = CVSS/10", **compare(cont), "distinct_scores_baseline": len(np.unique(BASE_P)), "distinct_scores_variant": len(np.unique(cont))}
print("(c) severity granularity:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in gran.items()})

sweep = []
for wk in (0.0, 0.25, 0.5, 1.0, 2.0):
    for we in (0.0, 0.5, 1.0, 2.0):
        sweep.append({"W_KEV": wk, "W_EPSS": we, **compare(portfolio_scores(kev=KEV, epss=EPSS, ext={"W_KEV": wk, "W_EPSS": we}))})
sweep_df = pd.DataFrame(sweep)

tt = oat.pivot(index="parameter", columns="direction", values="kendall_tau_b")
tt["worst_tau"] = tt.min(axis=1)
tt = tt.sort_values("worst_tau", ascending=False)
lo_col, hi_col = [c for c in tt.columns if c.startswith("-")][0], [c for c in tt.columns if c.startswith("+")][0]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].barh(tt.index, 1 - tt[lo_col], color="#4f81bd", label=lo_col); axes[0].barh(tt.index, -(1 - tt[hi_col]), color="#c0504d", label=hi_col)
axes[0].set(title=f"SIMULATED tornado: loss of rank agreement (1 - Kendall tau-b)\nwhen ONE weight changes by {DELTA:.0%}", xlabel="1 - tau-b   (blue: weight lowered, red: raised)"); axes[0].legend(fontsize=8)
axes[1].hist(mc_df["kendall_tau_b"], bins=30, alpha=0.7, label="Kendall tau-b", color="#5b9a5b"); axes[1].hist(mc_df["queue_jaccard"], bins=30, alpha=0.7, label=f"top-{CONFIG['TOP_K_FRACTION']:.0%} queue Jaccard", color="#d08a3b")
axes[1].set(title=f"SIMULATED Monte-Carlo: all weights jointly +-{DELTA:.0%} ({CONFIG['N_MONTE_CARLO']} draws)", xlabel="agreement with baseline ranking", ylabel="draws"); axes[1].legend()
sns.heatmap(sweep_df.pivot(index="W_KEV", columns="W_EPSS", values="queue_jaccard"), annot=True, fmt=".2f", cmap="viridis", ax=axes[2], cbar_kws={"label": f"top-{CONFIG['TOP_K_FRACTION']:.0%} queue Jaccard vs risk_model_v1"})
axes[2].set(title="SIMULATED: research exploit-weights vs queue stability\n(sensitivity only; not an accuracy measure)", xlabel="W_EPSS", ylabel="W_KEV")
savefig("R4_sensitivity_analysis")
SENS_SUMMARY = {"oat_most_sensitive_parameter": tt.index[-1], "oat_worst_tau": float(tt["worst_tau"].iloc[-1]), "mc_tau_p05": float(mc_df.kendall_tau_b.quantile(0.05)),
                "mc_jaccard_p05": float(mc_df.queue_jaccard.quantile(0.05)), "granularity": gran}
print(SENS_SUMMARY)

comp = pd.DataFrame({"CVSS-only": pd.Series(df["cvss3_band"].values[q_cvss]).value_counts(normalize=True),
                     "risk_model_v1 (portfolio)": pd.Series(df["cvss3_band"].values[q_base]).value_counts(normalize=True),
                     "risk_model_v1 + research exploit factor": pd.Series(df["cvss3_band"].values[queue_mask(portfolio_scores(kev=KEV, epss=EPSS, ext=EXPLOIT_EXT), CONFIG["TOP_K_FRACTION"])]).value_counts(normalize=True)}).reindex(band_order).fillna(0) * 100
display(comp.round(1).rename_axis("severity band of queued CVEs (%)"))

## 13. Supervised branch — runs **only if** the suitability gate passes
Without a legitimate labels file (the normal case with public data) this section prints why it is skipped and **no ML model is trained or exported**. If you supply `CONTEXT_LABELS_PATH`, the notebook:
1. applies **gate stage 2** — rejects labels that are (nearly) a deterministic function of `risk_model_v1` (Spearman ≥ `MAX_RULE_AGREEMENT`), since "ML vs baseline" would then be circular;
2. compares, on a held-out split, **(a) the independent `risk_model_v1` baseline, (b) the research exploit-aware variant, (c) Logistic Regression, (d) HistGradientBoosting** with *ranking* metrics — NDCG@k, Precision@k and Recall@k for the top priority class, Spearman — with bootstrap CIs on the NDCG difference to the strongest baseline;
3. reports permutation importance, confusion matrix and high-priority false negatives / false positives, then exports the model.
The baseline never sees the labels, and the ML models never see `risk_model_v1`'s output (they get the same raw inputs), so the comparison is fair.

In [ ]:
SUPERVISED = {"ran": False, "skipped_reasons": list(GATE_REASONS)}
if GATE_PASSED:
    from sklearn.calibration import calibration_curve
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.inspection import permutation_importance
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import confusion_matrix, ndcg_score
    from sklearn.model_selection import GroupShuffleSplit
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.utils.class_weight import compute_sample_weight

    lab = LABELS_RAW.copy()
    for c in ("asset_criticality", "asset_exposure", "attack_graph_evidence"):
        lab[c] = lab[c].astype(str).str.lower()
    lab["evidence_confidence"] = pd.to_numeric(lab["evidence_confidence"])
    feat = df.drop_duplicates("cve_id").set_index("cve_id")[["cvss3_score", "severity", "kev", "epss"]]
    n0 = len(lab); lab = lab.join(feat, on="cve_id", how="inner"); ledger("labels join", "finding's CVE not found in the (valid, v3-scored) NVD table", n0, len(lab))
    lab["v1"] = risk_v1(lab["severity"].values, lab["evidence_confidence"].values, lab["asset_criticality"].values, lab["asset_exposure"].values, lab["attack_graph_evidence"].values)
    lab["v1_ext"] = score_fast(code(lab["severity"], SEV_ORDER), lab["evidence_confidence"].values, code(lab["asset_criticality"], IMP_ORDER), code(lab["asset_exposure"], IMP_ORDER),
                               code(lab["attack_graph_evidence"], EV_ORDER), kev=lab["kev"].values.astype(float), epss=lab["epss"].values, ext=EXPLOIT_EXT)
    classes_sorted = sorted(lab["priority_label"].unique()); lab["y"] = lab["priority_label"].map({c: i for i, c in enumerate(classes_sorted)}).astype(int)
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.tree import DecisionTreeClassifier
    agreement = stats.spearmanr(lab["y"], lab["v1"]).correlation

    def rule_recovery(score):
        """Balanced accuracy with which thresholds on ONE rule score recover the labels (binned rule outputs lower Spearman but are still fully recoverable)."""
        t = DecisionTreeClassifier(max_leaf_nodes=len(classes_sorted) + 2, random_state=CONFIG["SEED"]).fit(np.asarray(score).reshape(-1, 1), lab["y"])
        return float(balanced_accuracy_score(lab["y"], t.predict(np.asarray(score).reshape(-1, 1))))

    recov = {"risk_model_v1": rule_recovery(lab["v1"]), "v1 + research exploit factor": rule_recovery(lab["v1_ext"])}
    print(f"GATE stage 2: Spearman(label, risk_model_v1) = {agreement:.3f} (limit {CONFIG['MAX_RULE_AGREEMENT']}); balanced accuracy of recovering labels from one rule score by thresholds: "
          f"{ {k: round(v, 3) for k, v in recov.items()} } (limit {CONFIG['MAX_RULE_BALANCED_ACCURACY']})")
    circular = agreement >= CONFIG["MAX_RULE_AGREEMENT"] or max(recov.values()) >= CONFIG["MAX_RULE_BALANCED_ACCURACY"]
    if circular:
        GATE_PASSED = False
        GATE_REASONS.append(f"labels are (nearly) a function of the rule: Spearman {agreement:.3f} (limit {CONFIG['MAX_RULE_AGREEMENT']}) / threshold-recovery balanced accuracy {max(recov.values()):.3f} "
                            f"(limit {CONFIG['MAX_RULE_BALANCED_ACCURACY']}); ML-vs-baseline would be circular")
        SUPERVISED["skipped_reasons"] = list(GATE_REASONS)
        print("GATE FAILED at stage 2:", GATE_REASONS[-1])

if GATE_PASSED:
    lab["crit_o"], lab["exp_o"], lab["sev_o"], lab["gr_o"] = code(lab["asset_criticality"], IMP_ORDER), code(lab["asset_exposure"], IMP_ORDER), code(lab["severity"], SEV_ORDER), code(lab["attack_graph_evidence"], EV_ORDER)
    FEATS = ["cvss3_score", "sev_o", "kev", "epss", "crit_o", "exp_o", "evidence_confidence", "gr_o"]
    if "labeled_at" in lab.columns:
        lab = lab.sort_values("labeled_at").reset_index(drop=True); i1, i2 = int(len(lab) * 0.70), int(len(lab) * 0.85)
        tr_i, va_i, te_i = np.arange(0, i1), np.arange(i1, i2), np.arange(i2, len(lab)); split_kind = "chronological by labeled_at"
    else:
        a, te_i = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=CONFIG["SEED"]).split(lab, groups=lab["cve_id"]))
        b, c = next(GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=CONFIG["SEED"]).split(lab.iloc[a], groups=lab.iloc[a]["cve_id"]))
        tr_i, va_i = a[b], a[c]; split_kind = "random grouped by cve_id (no timestamp supplied)"
    K = max(int(round(CONFIG["TOP_K_FRACTION"] * len(te_i))), 5); top_class = lab["y"].max()
    print(f"split: {split_kind}; train/val/test = {len(tr_i)}/{len(va_i)}/{len(te_i)}; k = {K}; top priority class = {top_class} ({int((lab.y == top_class).sum())} rows)")

    def rank_eval(y, s, k=K):
        y, s = np.asarray(y), np.asarray(s, float); o = np.argsort(-s, kind="stable")[:k]
        return {"ndcg@k": float(ndcg_score(y[None, :], s[None, :], k=k)), "precision@k(top class)": float(np.mean(y[o] == top_class)),
                "recall@k(top class)": float(np.sum(y[o] == top_class) / max(np.sum(y == top_class), 1)), "spearman": float(stats.spearmanr(y, s).correlation)}

    def make_model(kind, **kw):
        if kind == "logistic_regression":
            return Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("model", LogisticRegression(C=kw.get("C", 1.0), class_weight="balanced", max_iter=1000, random_state=CONFIG["SEED"]))])
        return Pipeline([("imp", SimpleImputer(strategy="median")), ("model", HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, max_leaf_nodes=kw.get("max_leaf_nodes", 15), min_samples_leaf=20, early_stopping=False, random_state=CONFIG["SEED"]))])

    def fit(kind, idx, **kw):
        extra = {"model__sample_weight": compute_sample_weight("balanced", lab["y"].iloc[idx])} if kind == "hist_gradient_boosting" else {}
        return make_model(kind, **kw).fit(lab.iloc[idx][FEATS], lab["y"].iloc[idx], **extra)

    def expected_priority(p, X):
        return p.predict_proba(X) @ p.classes_.astype(float)

    GRID = {"logistic_regression": [{"C": c} for c in (0.1, 1.0, 10.0)], "hist_gradient_boosting": [{"max_leaf_nodes": m} for m in (7, 15, 31)]}
    SUP_TUNE, FITTED_SUP, VAL_SCORE = [], {}, {}
    for kind, grid in GRID.items():
        best = None
        for prm in grid:
            m = fit(kind, tr_i, **prm); v = rank_eval(lab["y"].iloc[va_i], expected_priority(m, lab.iloc[va_i][FEATS]))["ndcg@k"]
            SUP_TUNE.append({"model": kind, "params": json.dumps(prm), "val_ndcg@k": v})
            if best is None or v > best[0]:
                best = (v, prm, m)
        FITTED_SUP[kind], VAL_SCORE[kind] = (best[2], best[1]), best[0]
    yte = lab["y"].iloc[te_i].values
    scores = {"random order (floor)": np.random.default_rng(CONFIG["SEED"]).random(len(te_i)), "risk_model_v1 baseline": lab["v1"].iloc[te_i].values, "v1 + research exploit factor": lab["v1_ext"].iloc[te_i].values,
              "severity only (CVSS)": lab["cvss3_score"].iloc[te_i].values}
    for kind, (m, prm) in FITTED_SUP.items():
        scores[kind] = expected_priority(m, lab.iloc[te_i][FEATS])
    sup_df = pd.DataFrame([{"scorer": nm, "uses_labels_for_fitting": nm in FITTED_SUP, **rank_eval(yte, s)} for nm, s in scores.items()]).sort_values("ndcg@k", ascending=False).reset_index(drop=True)
    display(sup_df.round(4))
    best_ml = max(VAL_SCORE, key=VAL_SCORE.get)   # chosen on VALIDATION
    best_base = max(["risk_model_v1 baseline", "v1 + research exploit factor", "severity only (CVSS)"], key=lambda k_: rank_eval(yte, scores[k_])["ndcg@k"])   # strongest baseline, chosen on test => favours the baseline
    rb = np.random.default_rng(CONFIG["SEED"]); diffs = []
    for _ in range(CONFIG["N_BOOTSTRAP"]):
        ix = rb.integers(0, len(yte), len(yte))
        if np.unique(yte[ix]).size > 1:
            diffs.append(rank_eval(yte[ix], scores[best_ml][ix])["ndcg@k"] - rank_eval(yte[ix], scores[best_base][ix])["ndcg@k"])
    diff_ci = (float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5)))
    ML_BEATS = diff_ci[0] > 0
    print(f"best ML (validation): {best_ml}; strongest baseline: {best_base}; NDCG@k difference ML - baseline: {np.mean(diffs):+.4f} (95% bootstrap CI {diff_ci[0]:+.4f} .. {diff_ci[1]:+.4f}) -> ML beats baseline: {ML_BEATS}")

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
    sns.barplot(data=sup_df, y="scorer", x="ndcg@k", hue="uses_labels_for_fitting", ax=axes[0], dodge=False); axes[0].set(title=f"Ranking quality on held-out labels (NDCG@{K})", xlabel="NDCG@k")
    sns.barplot(data=sup_df, y="scorer", x="recall@k(top class)", ax=axes[1], color="#5b9a5b"); axes[1].set(title=f"Recall@{K} of the top priority class", xlabel="recall@k")
    cmx = confusion_matrix(yte, FITTED_SUP[best_ml][0].predict(lab.iloc[te_i][FEATS]), labels=sorted(lab.y.unique()))
    sns.heatmap(cmx, annot=True, fmt="d", cmap="Blues", ax=axes[2]); axes[2].set(title=f"Confusion matrix: {best_ml}, test", xlabel="predicted priority class", ylabel="labelled priority class")
    savefig("S1_supervised_ranking_vs_baseline")
    mdl = FITTED_SUP[best_ml][0]
    pi = permutation_importance(mdl, lab.iloc[va_i][FEATS], lab["y"].iloc[va_i], scoring=lambda e, X, y: rank_eval(y, expected_priority(e, X))["ndcg@k"], n_repeats=5, random_state=CONFIG["SEED"], n_jobs=1)
    sup_pi = pd.DataFrame({"feature": FEATS, "ndcg_drop": pi.importances_mean, "sd": pi.importances_std}).sort_values("ndcg_drop", ascending=False)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    axes[0].barh(sup_pi["feature"][::-1], sup_pi["ndcg_drop"][::-1], xerr=sup_pi["sd"][::-1], color="#3b6ea5"); axes[0].set(title="Permutation importance (validation, NDCG@k drop)", xlabel="drop in NDCG@k")
    if top_class in mdl.classes_ and (yte == top_class).sum() >= 6:
        pbin = mdl.predict_proba(lab.iloc[te_i][FEATS])[:, list(mdl.classes_).index(top_class)]
        fr, mp = calibration_curve((yte == top_class).astype(int), pbin, n_bins=5, strategy="quantile"); axes[1].plot(mp, fr, "o-"); axes[1].plot([0, 1], [0, 1], "k:")
        axes[1].set(title=f"Reliability of P(top class) from {best_ml} (uncalibrated)", xlabel="mean predicted probability", ylabel="observed fraction")
    else:
        axes[1].axis("off")
    savefig("S2_supervised_importance_and_reliability")
    te_lab = lab.iloc[te_i].assign(ml=scores[best_ml], base=scores[best_base])
    thr_ml, thr_b = np.sort(te_lab["ml"])[::-1][K - 1], np.sort(te_lab["base"])[::-1][K - 1]
    fn_ml = te_lab[(te_lab.y == top_class) & (te_lab.ml < thr_ml)]; fp_ml = te_lab[(te_lab.y < top_class) & (te_lab.ml >= thr_ml)]; fn_b = te_lab[(te_lab.y == top_class) & (te_lab.base < thr_b)]
    print(f"high-priority findings missed by the top-{K} queue: ML {len(fn_ml)} (and {len(fp_ml)} lower-priority findings queued) vs baseline {len(fn_b)}")
    display(fn_ml.head(8)[["finding_id", "cve_id", "priority_label", "asset_criticality", "asset_exposure", "severity", "kev", "ml", "base"]].round(3))
    SUPERVISED.update(ran=True, split=split_kind, n_rows=len(lab), best_ml=best_ml, best_baseline=best_base, ml_beats_baseline=bool(ML_BEATS), ndcg_diff=float(np.mean(diffs)), ndcg_diff_ci95=diff_ci)
else:
    print("SUPERVISED BRANCH SKIPPED -- no legitimate contextual labels.")
    for r in GATE_REASONS:
        print("  -", r)
    print("No ML model is trained or exported by this notebook. What would be needed is listed in §14.")

## 14. Model limitations, intended use, and the data needed before any supervised model
**Intended use:** (a) explain how `risk_model_v1` behaves and how fragile its rankings are to its judgement-call weights; (b) provide a *fair evaluation harness* for the day legitimate labels exist; (c) inform a decision on whether to add exploit signals (KEV/EPSS) to a future `risk_model_v2`.

**Not valid:** treating any SIMULATED number as real-world performance; calling the deterministic baseline "validated" (ADR-019 says it is a transparent heuristic, *not scientifically validated*); training on labels derived from the rule, from KEV/EPSS or from invented asset values; presenting a rank correlation with CVSS as accuracy.

**Observed weakness worth acting on regardless of ML:** `risk_model_v1` maps severity to **five discrete weights**, so many findings share identical scores and top-k queues are unstable under small weight changes (§12) — a design fact about the heuristic, independent of any dataset.

**What data would let a supervised model be trained and validated later** (collect now via the audited analyst workflow — ADR-009's stated plan):

| Needed | Why | Minimum for a credible first study |
|---|---|---|
| Per-finding **analyst priority decisions** (accept/fix-now/defer/false-positive, with reason), or **incident outcomes** | the only honest ground truth for "what should be prioritised" | ≥ 500 labelled findings, ≥ 2 classes with ≥ 30 each; several organisations if generalisation is claimed |
| **Real asset context** at decision time (criticality, exposure, data sensitivity) | the very signal that distinguishes environmental risk from CVSS | recorded *when the decision was made*, not reconstructed |
| **Label provenance + timestamps + annotator id** | enables chronological splits, inter-annotator agreement, drift checks | ≥ 2 annotators on a subset to measure agreement (the ceiling for any model) |
| **Attack-graph evidence level** and finding **confidence/validation status** from the real pipeline | the other two inputs of the engine | taken from stored `risk_breakdown` rows |
| A frozen **evaluation protocol** (ranking metrics, chronological hold-out, baseline = shipped `risk_model_v1`) | prevents the circularity trap | pre-registered before looking at results |

## 15. Artifact export and reload verification
**Always exported:** the deterministic scoring configuration (with golden cases and parity status), scenario archetypes (flagged SIMULATED), sensitivity results, the suitability-gate report and an explicit statement of whether any ML model was trained. **Exported only if the supervised branch ran:** the model bundle and schema. The smoke test reloads the exported configuration from disk, rebuilds the scorer from it, and checks it against the golden cases and the in-notebook scorer.

In [ ]:
MODEL_DIR = OUTPUT_DIR / "models"; MODEL_DIR.mkdir(exist_ok=True)
ML_TRAINED = bool(SUPERVISED.get("ran"))
CONFIG_EXPORT = {
    "artifact": "deterministic risk scoring configuration (research copy of risk_model_v1 + labelled research extension)",
    "status": "EXPERIMENTAL / research -- the authoritative implementation is backend/src/offensivex/risk/engine.py; this file is not loaded by the application",
    "ml_model_trained": ML_TRAINED,
    "risk_tables": RISK_TABLES,
    "formula": "risk = clamp(SEV_W[severity]*10 * confidence * ((IMPACT_W[criticality]+IMPACT_W[exposure])/2) * (1 + graph_bonus_coefficient*EVID_W[weakest graph evidence]), 0, 10); graph term 1.0 if not on the attack graph",
    "research_exploit_extension": {**EXPLOIT_EXT, "formula": "risk_ext = clamp(risk_raw * (1 + W_KEV*kev + W_EPSS*epss), 0, 10)"},
    "golden_cases": [{"inputs": dict(zip(["severity", "confidence", "criticality", "exposure", "graph_evidence"], a)), "expected": e} for a, e in GOLDEN],
    "repository_parity": {"checked": PARITY_OK is not None, "ok": PARITY_OK, "message": PARITY_MSG},
    "scenario_archetypes_SIMULATED": SCENARIO_ARCHETYPES,
    "severity_mapping": CVSS_TO_OFFENSIVEX_SEVERITY,
}
write_json(OUTPUT_DIR / "risk_scoring_config.json", CONFIG_EXPORT)
write_json(OUTPUT_DIR / "suitability_gate.json", {"passed": bool(GATE_PASSED), "reasons": GATE_REASONS, "labels_file": Path(CONFIG["CONTEXT_LABELS_PATH"]).name if CONFIG["CONTEXT_LABELS_PATH"] else None,
                                                  "public_columns_considered": candidate_labels.to_dict(orient="records"), "ml_model_trained": ML_TRAINED})
write_json(OUTPUT_DIR / "metrics.json", {"fast_dev": CONFIG["FAST_DEV"], "ml_model_trained": ML_TRAINED, "observed_descriptive": OBSERVED, "simulated_sensitivity_summary": SENS_SUMMARY, "supervised": SUPERVISED,
                                        "NOTE": "all scenario/sensitivity numbers are SIMULATED and carry no information about real-world predictive performance"})
tables = {"scenario_ranking_agreement_SIMULATED": agree_df, "scenario_reversals_SIMULATED": rev.reset_index(names="medium_cve_on"), "sensitivity_one_at_a_time_SIMULATED": oat,
          "sensitivity_monte_carlo_SIMULATED": mc_df, "sensitivity_exploit_weights_SIMULATED": sweep_df, "queue_composition_SIMULATED": comp.reset_index(), "severity_vs_exploit_signals_OBSERVED": by_band.reset_index(),
          "gate_candidate_columns": candidate_labels}
if ML_TRAINED:
    tables.update({"supervised_ranking_comparison": sup_df, "supervised_tuning_log": pd.DataFrame(SUP_TUNE), "supervised_permutation_importance": sup_pi})
    joblib.dump({"pipeline": FITTED_SUP[best_ml][0], "features": FEATS, "classes": [int(c) for c in classes_sorted], "ml_beats_baseline": bool(ML_BEATS), "sklearn_version": sklearn.__version__}, MODEL_DIR / "contextual_priority_model.joblib", compress=3)
    write_json(OUTPUT_DIR / "feature_schema.json", {"inputs": {"cvss3_score": "float 0-10", "sev_o": "int index in " + str(SEV_ORDER), "kev": "0/1", "epss": "float 0-1 (NaN ok)", "crit_o": "int index in " + str(IMP_ORDER),
                                                                "exp_o": "int index in " + str(IMP_ORDER), "evidence_confidence": "float 0-1", "gr_o": "int index in " + str(EV_ORDER)}, "target": "ordinal priority class (see label_mapping)",
                                                    "label_mapping": {int(i): c for i, c in enumerate(classes_sorted)}, "output": "expected priority = sum_k k*P(class k); use for RANKING only",
                                                    "status": "EXPERIMENTAL; trained on user-supplied labels whose provenance is user-attested"})
export_common(tables=tables, extra_config={"fast_dev": CONFIG["FAST_DEV"], "ml_model_trained": ML_TRAINED, "labels_file": Path(CONFIG["CONTEXT_LABELS_PATH"]).name if CONFIG["CONTEXT_LABELS_PATH"] else None})

# ---- reload verification: rebuild the scorer purely from the exported JSON ----
cfg_loaded = json.loads((OUTPUT_DIR / "risk_scoring_config.json").read_text(encoding="utf-8"))
T_loaded = cfg_loaded["risk_tables"]
for case in cfg_loaded["golden_cases"]:
    i = case["inputs"]
    got = float(risk_v1([i["severity"]], [i["confidence"]], [i["criticality"]], [i["exposure"]], [i["graph_evidence"]], tables=T_loaded)[0])
    assert abs(got - case["expected"]) <= 0.006, f"reloaded config fails golden case {i}: {got} vs {case['expected']}"
rs_ = np.random.default_rng(7); m_ = 1000
lv = lambda order: [order[j] for j in rs_.integers(0, len(order), m_)]
sev_s, cr_s, ex_s, gr_s, cf_s = lv(SEV_ORDER), lv(IMP_ORDER), lv(IMP_ORDER), lv(EV_ORDER), rs_.uniform(0.05, 1, m_)
assert np.array_equal(risk_v1(sev_s, cf_s, cr_s, ex_s, gr_s, tables=T_loaded), risk_v1(sev_s, cf_s, cr_s, ex_s, gr_s)), "reloaded tables disagree with in-notebook tables"
assert set(np.unique(risk_band(risk_v1(sev_s, cf_s, cr_s, ex_s, gr_s)))) <= {"low", "medium", "high", "critical"}
print(f"SMOKE TEST PASSED: scoring config reloaded from disk; {len(cfg_loaded['golden_cases'])} golden cases and {m_:,} random inputs reproduce the in-notebook scorer; repository parity: {PARITY_MSG[:60]}")
if ML_TRAINED:
    bundle = joblib.load(MODEL_DIR / "contextual_priority_model.joblib")
    assert np.allclose(expected_priority(bundle["pipeline"], lab.iloc[te_i][FEATS].head(5)), scores[best_ml][:5]), "reloaded supervised model disagrees"
    print("supervised model bundle reloaded and verified.")
else:
    print("NO ML MODEL ARTIFACT WAS PRODUCED (gate not satisfied) -- by design; see suitability_gate.json.")
SMOKE_OK = True

## 16. Reproducibility checklist and conclusions
Generated from this run's numbers. **Simulated results are labelled as such and say nothing about real-world risk.**

In [ ]:
display(pd.DataFrame([("random seed", CONFIG["SEED"]), ("dataset fingerprint", DATA_META["dataset_fingerprint_sha256"][:24] + "..."), ("source", DATA_META["source_kind"]), ("analysed CVEs", f"{len(df):,}"),
                      ("contextual labels supplied", bool(CONFIG["CONTEXT_LABELS_PATH"])), ("suitability gate passed", bool(GATE_PASSED)), ("ML model trained", ML_TRAINED),
                      ("risk_model_v1 repository parity", PARITY_MSG[:70]), ("python / sklearn", f"{DEPENDENCY_VERSIONS['python']} / {DEPENDENCY_VERSIONS['sklearn']}"),
                      ("FAST_DEV (must be False to report)", CONFIG["FAST_DEV"]), ("plots saved", len(SAVED_PLOTS)), ("reload smoke test", SMOKE_OK), ("output dir", str(OUTPUT_DIR))], columns=["item", "value"]))
g = SENS_SUMMARY["granularity"]
lines = [
    f"* **RQ1 (gate).** {'PASSED' if GATE_PASSED else 'FAILED'}: " + ("; ".join(GATE_REASONS) if GATE_REASONS else "labels file satisfied all checks") + (" -> **no ML model was trained**; the deterministic baseline, simulated scenarios and sensitivity analysis were produced instead." if not ML_TRAINED else " -> supervised comparison executed (§13)."),
    f"* **RQ2 (OBSERVED).** Severity and exploit evidence are only weakly aligned: Spearman(CVSS, EPSS) = {OBSERVED['spearman_cvss_epss']:.2f}; {OBSERVED['kev_share_low_or_medium']:.0%} of KEV CVEs are only LOW/MEDIUM and {OBSERVED['critical_share_epss_lt_0_01']:.0%} of CRITICAL CVEs have EPSS < 0.01. A severity-only top-{CONFIG['TOP_K_FRACTION']:.0%} queue contains {OBSERVED['kev_in_cvss_topk']:.0%} of KEV CVEs (chance {CONFIG['TOP_K_FRACTION']:.0%}). This is descriptive, not a validation of any scorer.",
    f"* **RQ3 (SIMULATED).** With hypothetical contexts, `risk_model_v1` reorders CVSS-only rankings substantially (Kendall tau-b vs CVSS-only between {agree_df['kendall_tau_b_vs_CVSS_only'].min():.2f} and {agree_df['kendall_tau_b_vs_CVSS_only'].max():.2f} across archetypes; queue overlap with the CVSS queue {agree_df['queue_overlap_with_CVSS_queue (Jaccard)'].min():.2f}-{agree_df['queue_overlap_with_CVSS_queue (Jaccard)'].max():.2f}); a MEDIUM CVE on a critical, exposed asset can outrank a CRITICAL CVE on a low-value internal one -- by construction of the formula, not as evidence about reality.",
    f"* **RQ4 (SIMULATED sensitivity).** Rank correlations are fairly stable (one-at-a-time worst tau-b {SENS_SUMMARY['oat_worst_tau']:.2f} for `{SENS_SUMMARY['oat_most_sensitive_parameter']}`; Monte-Carlo 5th-percentile tau-b {SENS_SUMMARY['mc_tau_p05']:.2f}) but **top-{CONFIG['TOP_K_FRACTION']:.0%} queue membership is not** (5th-percentile Jaccard {SENS_SUMMARY['mc_jaccard_p05']:.2f}) because five severity weights give only {g['distinct_scores_baseline']} distinct portfolio scores (continuous severity: {g['distinct_scores_variant']}). Heavy ties make 'top-k' ill-defined under small weight changes.",
    ("* **RQ5 (supervised, USER-SUPPLIED labels).** " + (f"best ML `{SUPERVISED['best_ml']}` vs strongest baseline `{SUPERVISED['best_baseline']}`: NDCG@k difference {SUPERVISED['ndcg_diff']:+.3f} (95% CI {SUPERVISED['ndcg_diff_ci95'][0]:+.3f} .. {SUPERVISED['ndcg_diff_ci95'][1]:+.3f}); ML beats the baseline: {SUPERVISED['ml_beats_baseline']}. Valid only for the supplied labels' provenance and split ({SUPERVISED['split']})." if ML_TRAINED else "not executed (gate not satisfied). Data required: see §14.")),
    "* **Decision for OffensiveX:** keep `risk_model_v1` as the authoritative engine. Candidate follow-ups that need no ML: (1) consider continuous severity or a tie-break to avoid queue instability; (2) decide whether KEV/EPSS belong in a future `risk_model_v2` (an ADR + weights chosen by stakeholders); (3) start capturing analyst triage decisions with real asset context so a supervised comparison becomes possible (ADR-009).",
]
display(Markdown("\n".join(lines)))
report = f"""# Experiment report — contextual risk prioritisation
*Generated {datetime.now(timezone.utc).isoformat(timespec='seconds')} by notebooks/03_contextual_risk_prioritization.ipynb. fast_dev={CONFIG['FAST_DEV']}.*

**ML model trained: {'YES (user-supplied labels)' if ML_TRAINED else 'NO -- no validated ML model was trained (suitability gate not satisfied).'}**

Status: EXPERIMENTAL. `risk_model_v1` in the repository remains authoritative. All scenario/sensitivity numbers below are SIMULATED.

## Conclusions
{chr(10).join(lines)}
"""
(OUTPUT_DIR / "experiment_report.md").write_text(report, encoding="utf-8")
card = f"""# Model card — contextual risk prioritisation (deterministic scoring configuration)
**ML model trained:** {'yes, on user-supplied labels (see feature_schema.json)' if ML_TRAINED else '**no** -- the suitability gate was not satisfied, so no machine-learned model exists or is exported'}.

**What is exported:** a research copy of `risk_model_v1` (weights, formula, golden cases, repository-parity status), a labelled research exploit extension, hypothetical scenario archetypes (SIMULATED), sensitivity results.

**Intended use:** understanding and stress-testing the heuristic; evaluation harness for future labelled data.

**Prohibited interpretations:** treating simulated results as real-world performance; calling the heuristic validated; training on rule-derived, KEV/EPSS-derived or invented asset labels; replacing the repository engine with this file.

**Known limits:** weights are judgement calls (ADR-019); five severity weights create heavy score ties; the exploit extension is unvalidated; asset context in all scenarios is assumed.

**Data/secrets:** none included; dataset metadata holds hashes only.
"""
(OUTPUT_DIR / "model_card.md").write_text(card, encoding="utf-8")
scan_for_secrets(OUTPUT_DIR)
print(f"\nExported {sum(1 for p in OUTPUT_DIR.rglob('*') if p.is_file())} files to {OUTPUT_DIR}")
print("plots:", sorted(SAVED_PLOTS))